# Data Quality — Missing Value Validation

## 1. Introduction and Objectives

This notebook implements **Task 1 only: missing-value validation** across all nine Olist source datasets. It builds on `01_data_understanding.ipynb` by investigating *why* values are absent, whether the absence is expected or suspicious, and how missingness affects the two purchase-time ML objectives.

The authoritative prediction boundary is order placement (`order_purchase_timestamp`). Later lifecycle timestamps may be used to assess label availability, but not as model inputs. In particular:

- **Classification:** `is_late` requires valid actual and estimated customer-delivery timestamps.
- **Regression:** `delivery_days` requires valid purchase and actual customer-delivery timestamps.
- Reviews and post-purchase status/timestamps are unavailable at prediction time and are excluded from future features.

This is a read-only investigation. It does not clean, impute, remove, overwrite, or model any records. Duplicate validation, broad type checks, outliers, referential integrity, and comprehensive timestamp consistency belong to later tasks.

## 2. Load All Nine Datasets

**What are we checking?** That every expected source file is available and can be loaded with a stable, project-relative path.

**Why does it matter?** Missingness conclusions must come from the current CSV snapshot, not copied counts or hidden notebook state.

**What should we learn?** The shape table establishes the denominator used by every percentage below. Only the three lifecycle fields needed for order-missingness analysis are parsed as datetimes; the source files remain untouched.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:.4f}".format)


def find_project_root(start: Path) -> Path:
    '''Find the repository root from either the project or notebooks directory.'''
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project data/raw directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

DATASET_FILES = {
    "Orders": "olist_orders_dataset.csv",
    "Order Items": "olist_order_items_dataset.csv",
    "Products": "olist_products_dataset.csv",
    "Customers": "olist_customers_dataset.csv",
    "Sellers": "olist_sellers_dataset.csv",
    "Payments": "olist_order_payments_dataset.csv",
    "Reviews": "olist_order_reviews_dataset.csv",
    "Geolocation": "olist_geolocation_dataset.csv",
    "Category Translation": "product_category_name_translation.csv",
}


def load_source_csv(file_name: str) -> pd.DataFrame:
    '''Load one immutable source CSV and handle a possible UTF-8 BOM.'''
    file_path = RAW_DATA_DIR / file_name
    if not file_path.is_file():
        raise FileNotFoundError(f"Required source file not found: {file_path}")
    return pd.read_csv(file_path, encoding="utf-8-sig")


datasets = {
    dataset_name: load_source_csv(file_name)
    for dataset_name, file_name in DATASET_FILES.items()
}

orders = datasets["Orders"]
order_items = datasets["Order Items"]
products = datasets["Products"]
customers = datasets["Customers"]
sellers = datasets["Sellers"]
payments = datasets["Payments"]
reviews = datasets["Reviews"]
geolocation = datasets["Geolocation"]
category_translation = datasets["Category Translation"]

ORDER_DATETIME_COLUMNS = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]
for column in ORDER_DATETIME_COLUMNS:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

dataset_shapes = pd.DataFrame(
    [
        {
            "Dataset": dataset_name,
            "Source CSV": DATASET_FILES[dataset_name],
            "Rows": len(dataframe),
            "Columns": dataframe.shape[1],
        }
        for dataset_name, dataframe in datasets.items()
    ]
)
display(dataset_shapes)

,Dataset,Source CSV,Rows,Columns
0,Orders,olist_orders_dataset.csv,99441,8
1,Order Items,olist_order_items_dataset.csv,112650,7
2,Products,olist_products_dataset.csv,32951,9
3,Customers,olist_customers_dataset.csv,99441,5
4,Sellers,olist_sellers_dataset.csv,3095,4
5,Payments,olist_order_payments_dataset.csv,103886,5
6,Reviews,olist_order_reviews_dataset.csv,99224,7
7,Geolocation,olist_geolocation_dataset.csv,1000163,5
8,Category Translation,product_category_name_translation.csv,71,2


## 3. Missing Values Overview

**What are we checking?** Pandas `NaN` counts and percentages for every column, plus the number of rows containing at least one missing value.

**Why does it matter?** A missing-cell count and an affected-row count answer different questions. One row may contain several missing cells but should count only once in the row-level measure.

**What should we learn?** Percentages use each dataset's row count as the denominator. A nonzero result is an investigation prompt—not automatic proof of a defect—because a field can be optional, not yet applicable in a business process, or genuinely absent.

In [2]:
def missing_value_profile(
    dataset_name: str, dataframe: pd.DataFrame
) -> tuple[dict, pd.DataFrame]:
    '''Return dataset-level and per-column Pandas-NaN summaries.

    Percentages use the dataset row count. The helper reports evidence only and
    deliberately performs no filling, filtering, or source-data mutation.
    '''
    row_count = len(dataframe)
    missing_counts = dataframe.isna().sum()
    rows_with_missing = int(dataframe.isna().any(axis=1).sum())

    dataset_summary = {
        "Dataset": dataset_name,
        "Total Rows": row_count,
        "Column Names": ", ".join(dataframe.columns),
        "Rows With Missing": rows_with_missing,
        "Rows With Missing (%)": (
            rows_with_missing / row_count * 100 if row_count else 0.0
        ),
    }
    column_summary = pd.DataFrame(
        {
            "Dataset": dataset_name,
            "Column": dataframe.columns,
            "Missing Count": missing_counts.to_numpy(),
            "Missing Percentage": (
                missing_counts.to_numpy() / row_count * 100
                if row_count
                else 0.0
            ),
        }
    )
    return dataset_summary, column_summary


dataset_missing_rows = []
column_missing_tables = []
for dataset_name, dataframe in datasets.items():
    dataset_summary, column_summary = missing_value_profile(
        dataset_name, dataframe
    )
    dataset_missing_rows.append(dataset_summary)
    column_missing_tables.append(column_summary)

dataset_missing_overview = pd.DataFrame(dataset_missing_rows)
column_missing_overview = pd.concat(column_missing_tables, ignore_index=True)

display(dataset_missing_overview)
display(column_missing_overview)

,Dataset,Total Rows,Column Names,Rows With Missing,Rows With Missing (%)
0,Orders,99441,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivere...",2980,2.9968
1,Order Items,112650,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value",0,0.0000
2,Products,32951,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_phot...",611,1.8543
3,Customers,99441,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state",0,0.0000
4,Sellers,3095,"seller_id, seller_zip_code_prefix, seller_city, seller_state",0,0.0000
5,Payments,103886,"order_id, payment_sequential, payment_type, payment_installments, payment_value",0,0.0000
6,Reviews,99224,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation...",89385,90.0841
7,Geolocation,1000163,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state",0,0.0000
8,Category Translation,71,"product_category_name, product_category_name_english",0,0.0000


,Dataset,Column,Missing Count,Missing Percentage
0,Orders,order_id,0,0.0000
1,Orders,customer_id,0,0.0000
2,Orders,order_status,0,0.0000
3,Orders,order_purchase_timestamp,0,0.0000
4,Orders,order_approved_at,160,0.1609
5,Orders,order_delivered_carrier_date,1783,1.7930
6,Orders,order_delivered_customer_date,2965,2.9817
7,Orders,order_estimated_delivery_date,0,0.0000
8,Order Items,order_id,0,0.0000
9,Order Items,order_item_id,0,0.0000


**Interpretation guidance:** Compare `Rows With Missing` with the per-column counts. Orders and products can have related fields absent together; reviews can have one or both optional text fields absent. Datasets with zero Pandas `NaN` still need a hidden-missingness check for blank strings and placeholder tokens.

## 4. Orders — Missing Value Validation

**What are we checking?** Missing approval, carrier-handoff, and actual customer-delivery timestamps by `order_status`, including delivered orders missing their actual-delivery timestamp.

**Why does it matter?** Lifecycle events may legitimately be absent when an order has not reached that stage. However, `order_delivered_customer_date` is required to observe both ML targets, so a delivered-status row without it is suspicious and label-ineligible until investigated.

**What should we learn?** The status-conditioned tables separate expected process-related missingness from contradictory combinations. Status itself is post-purchase information and is used here only for retrospective validation—not as a purchase-time feature.

In [3]:
ORDER_MISSING_COLUMNS = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
]

orders_missing_summary = pd.DataFrame(
    {
        "Column": ORDER_MISSING_COLUMNS,
        "Missing Count": orders[ORDER_MISSING_COLUMNS].isna().sum().to_numpy(),
        "Missing Percentage": (
            orders[ORDER_MISSING_COLUMNS].isna().mean().mul(100).to_numpy()
        ),
    }
)
display(orders_missing_summary)

order_status_missing_counts = pd.concat(
    {
        column: pd.crosstab(
            orders["order_status"],
            orders[column].isna(),
        ).rename(columns={False: "Present", True: "Missing"})
        for column in ORDER_MISSING_COLUMNS
    },
    names=["Column", "order_status"],
).astype("Int64")
display(order_status_missing_counts)

,Column,Missing Count,Missing Percentage
0,order_approved_at,160,0.1609
1,order_delivered_carrier_date,1783,1.7930
2,order_delivered_customer_date,2965,2.9817


Present  Missing
Column                        order_status                  
order_approved_at             approved            2        0
                              canceled          484      141
                              created             0        5
                              delivered       96464       14
                              invoiced          314        0
                              processing        301        0
                              shipped          1107        0
                              unavailable       609        0
order_delivered_carrier_date  approved            0        2
                              canceled           75      550
                              created             0        5
                              delivered       96476        2
                              invoiced            0      314
                              processing          0      301
                              shipped          1107        0
                              unavailable         0      609
order_delivered_customer_date approved            0        2
                              canceled            6      619
                              created             0        5
                              delivered       96470        8
                              invoiced            0      314
                              processing          0      301
                              shipped             0     1107
                              unavailable         0      609

In [4]:
order_suspicion_masks = {
    "Delivered but approval timestamp missing": (
        orders["order_status"].eq("delivered")
        & orders["order_approved_at"].isna()
    ),
    "Delivered but carrier timestamp missing": (
        orders["order_status"].eq("delivered")
        & orders["order_delivered_carrier_date"].isna()
    ),
    "Delivered but customer-delivery timestamp missing": (
        orders["order_status"].eq("delivered")
        & orders["order_delivered_customer_date"].isna()
    ),
    "Non-delivered status but customer-delivery timestamp present": (
        orders["order_status"].ne("delivered")
        & orders["order_delivered_customer_date"].notna()
    ),
}

order_suspicious_summary = pd.DataFrame(
    [
        {"Condition": condition, "Record Count": int(mask.sum())}
        for condition, mask in order_suspicion_masks.items()
    ]
)
display(order_suspicious_summary)

suspicious_order_mask = pd.concat(
    order_suspicion_masks.values(), axis=1
).any(axis=1)
suspicious_order_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    *ORDER_MISSING_COLUMNS,
    "order_estimated_delivery_date",
]
display(orders.loc[suspicious_order_mask, suspicious_order_columns].head(10))

,Condition,Record Count
0,Delivered but approval timestamp missing,14
1,Delivered but carrier timestamp missing,2
2,Delivered but customer-delivery timestamp missing,8
3,Non-delivered status but customer-delivery timestamp present,6


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00


**Interpretation guidance:** Missing lifecycle timestamps among canceled, unavailable, created, invoiced, processing, approved, or shipped orders may reflect an event that never occurred or had not occurred in the extract; those orders are not automatically invalid. Delivered-status contradictions require investigation, and so do the canceled orders that nevertheless carry an actual customer-delivery timestamp: a timestamp-only eligibility rule would treat them as labelable, while a status-based rule would exclude them. That choice is deferred to target construction, but it must be made explicitly. Any row without an observed customer-delivery timestamp cannot receive either target label under the authoritative definitions; it must not be silently labeled on time or assigned a duration. Approval and carrier timestamps are post-purchase and are never candidate predictors here.

## 5. Products — Missing Value Validation

**What are we checking?** Whether descriptive catalog fields and physical measurements are missing individually or as coordinated groups.

**Why does it matter?** Co-missing fields can reveal a systematic catalog-ingestion pattern, while isolated physical missingness may indicate a small number of incomplete product records. Both can affect later feature availability.

**What should we learn?** Evidence about co-missingness should guide future treatment decisions. This notebook does not impute values or approve any feature for modeling.

In [5]:
PRODUCT_METADATA_COLUMNS = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
]
PRODUCT_PHYSICAL_COLUMNS = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
PRODUCT_INVESTIGATION_COLUMNS = (
    PRODUCT_METADATA_COLUMNS + PRODUCT_PHYSICAL_COLUMNS
)

product_missing_summary = pd.DataFrame(
    {
        "Column": PRODUCT_INVESTIGATION_COLUMNS,
        "Missing Count": products[PRODUCT_INVESTIGATION_COLUMNS]
        .isna()
        .sum()
        .to_numpy(),
        "Missing Percentage": products[PRODUCT_INVESTIGATION_COLUMNS]
        .isna()
        .mean()
        .mul(100)
        .to_numpy(),
    }
)
display(product_missing_summary)

,Column,Missing Count,Missing Percentage
0,product_category_name,610,1.8512
1,product_name_lenght,610,1.8512
2,product_description_lenght,610,1.8512
3,product_photos_qty,610,1.8512
4,product_weight_g,2,0.0061
5,product_length_cm,2,0.0061
6,product_height_cm,2,0.0061
7,product_width_cm,2,0.0061


In [6]:
product_missing_flags = products[PRODUCT_INVESTIGATION_COLUMNS].isna()
product_pattern_counts = (
    product_missing_flags.value_counts()
    .rename("Product Count")
    .reset_index()
    .sort_values("Product Count", ascending=False)
)
display(product_pattern_counts.head(10))

metadata_missing_count = product_missing_flags[PRODUCT_METADATA_COLUMNS].sum(axis=1)
physical_missing_count = product_missing_flags[PRODUCT_PHYSICAL_COLUMNS].sum(axis=1)

product_group_patterns = pd.DataFrame(
    [
        {
            "Pattern": "All four descriptive metadata fields missing",
            "Product Count": int(
                product_missing_flags[PRODUCT_METADATA_COLUMNS].all(axis=1).sum()
            ),
        },
        {
            "Pattern": "Some, but not all, descriptive fields missing",
            "Product Count": int(
                ((metadata_missing_count > 0) & (metadata_missing_count < 4)).sum()
            ),
        },
        {
            "Pattern": "All four physical attributes missing",
            "Product Count": int(
                product_missing_flags[PRODUCT_PHYSICAL_COLUMNS].all(axis=1).sum()
            ),
        },
        {
            "Pattern": "Some, but not all, physical attributes missing",
            "Product Count": int(
                ((physical_missing_count > 0) & (physical_missing_count < 4)).sum()
            ),
        },
        {
            "Pattern": "Both metadata and physical groups affected",
            "Product Count": int(
                ((metadata_missing_count > 0) & (physical_missing_count > 0)).sum()
            ),
        },
    ]
)
display(product_group_patterns)

product_missing_examples = products.loc[
    product_missing_flags.any(axis=1),
    ["product_id", *PRODUCT_INVESTIGATION_COLUMNS],
].head(10)
display(product_missing_examples)

,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,Product Count
0,False,False,False,False,False,False,False,False,32340
1,True,True,True,True,False,False,False,False,609
2,False,False,False,False,True,True,True,True,1
3,True,True,True,True,True,True,True,True,1


,Pattern,Product Count
0,All four descriptive metadata fields missing,610
1,"Some, but not all, descriptive fields missing",0
2,All four physical attributes missing,2
3,"Some, but not all, physical attributes missing",0
4,Both metadata and physical groups affected,1


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0000,17.0000,14.0000,12.0000
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0000,16.0000,7.0000,20.0000
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0000,20.0000,20.0000,20.0000
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0000,41.0000,30.0000,41.0000
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0000,35.0000,7.0000,12.0000
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0000,16.0000,2.0000,11.0000
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0000,16.0000,7.0000,11.0000
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0000,18.0000,8.0000,14.0000
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0000,22.0000,14.0000,14.0000
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0000,25.0000,25.0000,25.0000


**Interpretation guidance:** A dominant all-or-none pattern across category/name-length/description-length/photo fields is evidence of systematic descriptive-metadata absence rather than unrelated random cells. Compare that with the physical group: very few products missing every measurement indicates a separate, isolated issue. These fields may be purchase-time feature candidates only after serving availability is established; later preprocessing must preserve missingness information and learn any treatment from training data only.

## 6. Reviews — Missing Value Validation

**What are we checking?** Optional review title/message absence overall, by review score, and across text-presence combinations.

**Why does it matter?** Sparse optional text is different from a broken required field. Variation by score may also reveal how customers choose to leave written feedback.

**What should we learn?** Reviews are useful for retrospective experience analysis, but they occur after purchase and must not be used as purchase-time predictive features.

In [7]:
REVIEW_TEXT_COLUMNS = ["review_comment_title", "review_comment_message"]

review_missing_summary = pd.DataFrame(
    {
        "Column": REVIEW_TEXT_COLUMNS,
        "Missing Count": reviews[REVIEW_TEXT_COLUMNS].isna().sum().to_numpy(),
        "Missing Percentage": (
            reviews[REVIEW_TEXT_COLUMNS].isna().mean().mul(100).to_numpy()
        ),
    }
)
display(review_missing_summary)

review_missing_by_score = (
    reviews.groupby("review_score")[REVIEW_TEXT_COLUMNS]
    .agg(["count", lambda series: series.isna().sum(), lambda series: series.isna().mean() * 100])
)
review_missing_by_score.columns = [
    f"{column}_{metric}"
    for column, metric in review_missing_by_score.columns
]
review_missing_by_score = review_missing_by_score.rename(
    columns=lambda name: name.replace("_<lambda_0>", "_missing_count").replace(
        "_<lambda_1>", "_missing_percentage"
    )
)
display(review_missing_by_score)

,Column,Missing Count,Missing Percentage
0,review_comment_title,87656,88.3415
1,review_comment_message,58247,58.7025


,review_comment_title_count,review_comment_title_missing_count,review_comment_title_missing_percentage,review_comment_message_count,review_comment_message_missing_count,review_comment_message_missing_percentage
review_score,,,,,,
1,1873,9551,83.6047,8745,2679,23.4506
2,478,2673,84.8302,2145,1006,31.9264
3,824,7355,89.9254,3557,4622,56.5106
4,1735,17407,90.9362,5976,13166,68.7807
5,6658,50670,88.3861,20554,36774,64.1467


In [8]:
review_text_presence = pd.DataFrame(
    {
        "Title Present": reviews["review_comment_title"].notna(),
        "Message Present": reviews["review_comment_message"].notna(),
    }
)
review_text_combinations = (
    review_text_presence.value_counts()
    .rename("Review Count")
    .reset_index()
    .sort_values("Review Count", ascending=False)
)
review_text_combinations["Percentage"] = (
    review_text_combinations["Review Count"] / len(reviews) * 100
)
display(review_text_combinations)

review_text_by_score = pd.crosstab(
    reviews["review_score"],
    review_text_presence["Message Present"],
    normalize="index",
).rename(columns={False: "Without Message (%)", True: "With Message (%)"}) * 100
display(review_text_by_score)

,Title Present,Message Present,Review Count,Percentage
0,False,False,56518,56.9600
1,False,True,31138,31.3815
2,True,True,9839,9.9159
3,True,False,1729,1.7425


Message Present,Without Message (%),With Message (%)
review_score,,
1,23.4506,76.5494
2,31.9264,68.0736
3,56.5106,43.4894
4,68.7807,31.2193
5,64.1467,35.8533


**Interpretation guidance:** The presence/absence combinations show that a review can be valid without either text field; the numeric score and review timestamps remain populated. Score-level differences are behavioral evidence, not a reason to manufacture comments. All review fields describe post-purchase experience and therefore remain excluded from both purchase-time models.

## 7. Other Six Datasets — Hidden Missing Values

**What are we checking?** Pandas `NaN`, empty strings, whitespace-only strings, and a conservative set of common missing-value tokens in Customers, Sellers, Order Items, Payments, Geolocation, and Category Translation.

**Why does it matter?** CSV text can look populated to Pandas while carrying an empty or placeholder-like value. These categories must remain distinct because their meanings and remedies differ.

**Why read the files a second time?** By default `pd.read_csv` converts empty fields and tokens such as `NA`, `N/A`, `null`, `None`, `nan`, and `n/a` into `NaN` while loading. An empty-string or placeholder check on the already-loaded DataFrame can therefore never find those values: they have already become `NaN`. To see what is literally written in each file, we reload it with `dtype=str` and `keep_default_na=False`, which disables that conversion. The `Converted To NaN` column then shows whether Pandas turned any *non-empty* text into `NaN`.

**What should we learn?** A literal token is only a *suspected* placeholder until interpreted in its column context. For example, a documented business category such as `not_defined` is not automatically equivalent to `NaN`.

In [9]:
# Checked against the raw file text, so tokens that Pandas would normally
# convert to NaN on load (for example "NA" or "null") are still visible here.
COMMON_PLACEHOLDER_TOKENS = {"n/a", "na", "nan", "null", "none", "unknown", "?", "-"}
OTHER_DATASETS = {
    "Customers": customers,
    "Sellers": sellers,
    "Order Items": order_items,
    "Payments": payments,
    "Geolocation": geolocation,
    "Category Translation": category_translation,
}


def load_raw_text_csv(file_name: str) -> pd.DataFrame:
    '''Reload a source CSV as literal text with default NA conversion disabled.

    Every column is read as `str` and `keep_default_na=False` keeps empty fields
    as "" and tokens such as "NA" as text, exposing exactly what the file holds.
    '''
    return pd.read_csv(
        RAW_DATA_DIR / file_name,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )


def hidden_missing_profile(
    dataset_name: str,
    parsed: pd.DataFrame,
    raw: pd.DataFrame,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    '''Compare each column's Pandas NaN count with its raw-text representation.

    `parsed` is the default-loaded DataFrame and `raw` is the same file from
    `load_raw_text_csv`. Counts are evidence only; no values are changed.
    '''
    rows = []
    for column in columns or list(parsed.columns):
        raw_text = raw[column]
        stripped = raw_text.str.strip()
        parsed_nan = parsed[column].isna()
        rows.append(
            {
                "Dataset": dataset_name,
                "Column": column,
                "Pandas NaN": int(parsed_nan.sum()),
                "Empty String": int(raw_text.eq("").sum()),
                "Whitespace Only": int((stripped.eq("") & raw_text.ne("")).sum()),
                "Suspected Placeholder": int(
                    stripped.str.lower().isin(COMMON_PLACEHOLDER_TOKENS).sum()
                ),
                # Non-empty file text that Pandas silently turned into NaN on load.
                "Converted To NaN": int((parsed_nan & raw_text.ne("")).sum()),
            }
        )
    return pd.DataFrame(rows)


HIDDEN_MISSING_COLUMNS = [
    "Pandas NaN",
    "Empty String",
    "Whitespace Only",
    "Suspected Placeholder",
    "Converted To NaN",
]

hidden_missing_details = pd.concat(
    [
        hidden_missing_profile(
            dataset_name, dataframe, load_raw_text_csv(DATASET_FILES[dataset_name])
        )
        for dataset_name, dataframe in OTHER_DATASETS.items()
    ],
    ignore_index=True,
)
display(hidden_missing_details)

hidden_missing_by_dataset = hidden_missing_details.groupby(
    "Dataset", as_index=False
)[HIDDEN_MISSING_COLUMNS].sum()
display(hidden_missing_by_dataset)

# `not_defined` is a documented payment-type category, so it is reported for
# context but not relabeled as missing by the conservative token scan above.
not_defined_payment_count = int(
    payments["payment_type"].astype("string").str.strip().str.lower().eq("not_defined").sum()
)
pd.Series({"payment_type='not_defined' (context only)": not_defined_payment_count})


,Dataset,Column,Pandas NaN,Empty String,Whitespace Only,Suspected Placeholder,Converted To NaN
0,Customers,customer_id,0,0,0,0,0
1,Customers,customer_unique_id,0,0,0,0,0
2,Customers,customer_zip_code_prefix,0,0,0,0,0
3,Customers,customer_city,0,0,0,0,0
4,Customers,customer_state,0,0,0,0,0
5,Sellers,seller_id,0,0,0,0,0
6,Sellers,seller_zip_code_prefix,0,0,0,0,0
7,Sellers,seller_city,0,0,0,0,0
8,Sellers,seller_state,0,0,0,0,0
9,Order Items,order_id,0,0,0,0,0


,Dataset,Pandas NaN,Empty String,Whitespace Only,Suspected Placeholder,Converted To NaN
0,Category Translation,0,0,0,0,0
1,Customers,0,0,0,0,0
2,Geolocation,0,0,0,0,0
3,Order Items,0,0,0,0,0
4,Payments,0,0,0,0,0
5,Sellers,0,0,0,0,0


payment_type='not_defined' (context only)    3
dtype: int64

**Interpretation guidance:** Keep the four representations separate. `NaN` is Pandas' missing marker; an empty string has zero characters; a whitespace-only value contains characters but no visible content; and a placeholder token is merely suspicious until its semantics are verified. `Empty String` should equal `Pandas NaN` for each column (empty fields are what Pandas reads as `NaN`), and `Converted To NaN` should be zero; a nonzero value would mean real text was hidden by the default loader. No values are normalized or overwritten here.

### Review Text — Hidden Missing Values

**What are we checking?** The same raw-text scan applied to `review_comment_title` and `review_comment_message`, the free-text fields where blank or placeholder input is most likely.

**Why does it matter?** Section 6 treats any non-`NaN` comment as "text present". A comment containing only spaces, `?`, or `-` is not informative text, so those rows slightly understate effective text absence.

**What should we learn?** Whether the Section 6 missing percentages are exact or a lower bound. This remains retrospective analysis only; review data is never a purchase-time feature.

In [10]:
review_raw_text = load_raw_text_csv(DATASET_FILES["Reviews"])
review_hidden_missing = hidden_missing_profile(
    "Reviews", reviews, review_raw_text, REVIEW_TEXT_COLUMNS
)
display(review_hidden_missing)

# Show the literal values behind the whitespace/placeholder counts.
review_hidden_examples = pd.concat(
    {
        column: review_raw_text.loc[
            review_raw_text[column].ne("")
            & (
                review_raw_text[column].str.strip().eq("")
                | review_raw_text[column]
                .str.strip()
                .str.lower()
                .isin(COMMON_PLACEHOLDER_TOKENS)
            ),
            column,
        ]
        .map(repr)
        .value_counts()
        for column in REVIEW_TEXT_COLUMNS
    },
    names=["Column", "Raw Value"],
).rename("Review Count")
display(review_hidden_examples)


,Dataset,Column,Pandas NaN,Empty String,Whitespace Only,Suspected Placeholder,Converted To NaN
0,Reviews,review_comment_title,87656,87656,2,2,0
1,Reviews,review_comment_message,58247,58247,27,7,0


Column                  Raw Value                                 
review_comment_title    ' '                                            2
                        '?'                                            1
                        '-'                                            1
review_comment_message  '\r\n'                                        14
                        ' '                                            9
                        '\r\n\r\n'                                     3
                        '?'                                            3
                        '-'                                            3
                        'N/a'                                          1
                        '\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n'     1
Name: Review Count, dtype: int64

**Interpretation guidance:** These comments are populated in the CSV but carry no readable content. They are a small share of reviews, so the Section 6 conclusions about score-linked optional text do not change, but the reported message and title missing percentages should be read as a lower bound on *effective* text absence. Nothing is rewritten to `NaN` in this task.

## 8. Missingness Patterns

**What are we checking?** The strongest cross-field patterns from the focused investigations: coordinated product absence, order timestamps by status, and review-message absence by score.

**Why does it matter?** Missingness patterns can reveal business-process stages and data-collection mechanisms that isolated column totals hide.

**What should we learn?** Concise pattern tables are sufficient here; a chart would not add meaningful information beyond these exact counts and percentages.

In [11]:
order_customer_delivery_pattern = pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isna(),
).rename(columns={False: "Delivery Timestamp Present", True: "Delivery Timestamp Missing"})
order_customer_delivery_pattern["Missing Percentage Within Status"] = (
    order_customer_delivery_pattern.get("Delivery Timestamp Missing", 0)
    / order_customer_delivery_pattern.sum(axis=1)
    * 100
)

review_message_pattern = (
    reviews.groupby("review_score")["review_comment_message"]
    .agg(
        Reviews="size",
        Missing=lambda series: series.isna().sum(),
    )
)
review_message_pattern["Missing Percentage"] = (
    review_message_pattern["Missing"] / review_message_pattern["Reviews"] * 100
)

display(product_group_patterns)
display(order_customer_delivery_pattern)
display(review_message_pattern)

,Pattern,Product Count
0,All four descriptive metadata fields missing,610
1,"Some, but not all, descriptive fields missing",0
2,All four physical attributes missing,2
3,"Some, but not all, physical attributes missing",0
4,Both metadata and physical groups affected,1


order_delivered_customer_date,Delivery Timestamp Present,Delivery Timestamp Missing,Missing Percentage Within Status
order_status,,,
approved,0,2,100.0000
canceled,6,619,99.0400
created,0,5,100.0000
delivered,96470,8,0.0083
invoiced,0,314,100.0000
processing,0,301,100.0000
shipped,0,1107,100.0000
unavailable,0,609,100.0000


,Reviews,Missing,Missing Percentage
review_score,,,
1,11424,2679,23.4506
2,3151,1006,31.9264
3,8179,4622,56.5106
4,19142,13166,68.7807
5,57328,36774,64.1467


**Interpretation guidance:** Treat these as mechanisms to investigate, not causal proof. Status-linked timestamp absence is consistent with lifecycle progression; coordinated product metadata absence is consistent with a catalog-record pattern; score-linked review text reflects optional response behavior. None of these observations authorizes deletion or imputation.

## 9. ML Impact Assessment

Missingness has different consequences depending on the field's role and availability at order placement.

| Missingness role | Classification impact | Regression impact | Purchase-time interpretation |
|---|---|---|---|
| Actual customer-delivery timestamp | Required to observe `is_late`; missing means no supervised label | Required to calculate `delivery_days`; missing means no supervised label | Outcome-only; never a feature |
| Estimated delivery timestamp | Required for `is_late`; no missing values are observed in this snapshot | Not required for the duration formula, though potentially useful as a promise feature | Intended purchase-time candidate; serving availability still needs validation |
| Purchase timestamp | Needed for regression target start and chronological splitting; no missing values are observed | Required to calculate `delivery_days` | Prediction boundary and potential calendar-feature source |
| Approval/carrier timestamps | Not needed for the target definition | Not needed for the target definition | Post-purchase; missingness may be process-related, but values are prohibited features |
| Product catalog metadata and measurements | May reduce availability of candidate product features | Same potential feature impact | Candidate features only; treatment must be learned later from training data |
| Review title/message | No target requirement and no permitted feature use | No target requirement and no permitted feature use | Post-purchase leakage; retrospective analysis only |
| Status/date contradiction (canceled orders with an actual delivery timestamp) | Would pass a timestamp-only `is_late` eligibility rule but fail a status-based rule | Same ambiguity for `delivery_days` | Eligibility rule is an open decision for target construction; never a feature |
| Expected lifecycle absence | May explain why an order lacks an observed outcome | May explain why an order lacks an observed outcome | Do not equate an unfinished/canceled process with corrupt data |

The supervised training cohort will be narrower than the deployment population because some historical orders lack observable targets. Eligibility must be assessed explicitly in a later target-construction task; missing outcomes must never be converted silently to negative labels or zero duration.

## 10. Findings and Recommendations

**What are we checking?** A consolidated, evidence-linked decision table for every field or dataset investigated in this task.

**Why does it matter?** Column-level totals can mix legitimate and suspicious cases. Classifications below are therefore conditional where status or field group changes the interpretation.

**What should we learn?** Recommendations are investigative and eligibility-oriented. They do not perform cleaning or prescribe an unvalidated imputation strategy.

In [12]:
def missing_count_and_percentage(
    dataframe: pd.DataFrame, column: str
) -> tuple[int, float]:
    '''Return a column's current missing count and row-based percentage.'''
    count = int(dataframe[column].isna().sum())
    percentage = count / len(dataframe) * 100 if len(dataframe) else 0.0
    return count, percentage


canceled_with_delivery_count = int(
    (
        orders["order_status"].eq("canceled")
        & orders["order_delivered_customer_date"].notna()
    ).sum()
)

findings_rows = []

for column in ORDER_MISSING_COLUMNS:
    count, percentage = missing_count_and_percentage(orders, column)
    delivered_missing = int(
        (orders["order_status"].eq("delivered") & orders[column].isna()).sum()
    )
    if column == "order_delivered_customer_date":
        ml_impact = "Classification and regression target eligibility"
        action = "Assess label eligibility; investigate delivered-status and canceled-with-delivery contradictions"
    else:
        ml_impact = "None as a feature; lifecycle-quality context only"
        action = "Retain for audit; investigate delivered-status contradictions"
    findings_rows.append(
        {
            "Dataset": DATASET_FILES["Orders"],
            "Column": column,
            "Missing Count": count,
            "Missing Percentage": percentage,
            "Percentage Basis": "Rows",
            "Classification": "Expected by some statuses / Needs Investigation when delivered",
            "Evidence": (
                f"Missingness varies by order_status; {delivered_missing} delivered rows are missing this field."
            ),
            "ML Impact": ml_impact,
            "Proposed Action": action,
        }
    )

# A contradiction in the opposite direction: an outcome timestamp exists even
# though the status says the order was canceled. This affects label eligibility.
findings_rows.append(
    {
        "Dataset": DATASET_FILES["Orders"],
        "Column": "order_status vs order_delivered_customer_date",
        "Missing Count": canceled_with_delivery_count,
        "Missing Percentage": canceled_with_delivery_count / len(orders) * 100,
        "Percentage Basis": "Rows",
        "Classification": "Needs Investigation",
        "Evidence": (
            f"{canceled_with_delivery_count} canceled orders have an actual customer-delivery timestamp (count of contradictions, not missing cells)."
        ),
        "ML Impact": "Open target-eligibility decision: timestamp-only vs status-based rule",
        "Proposed Action": "Decide and document the eligibility rule in target construction; report the count",
    }
)

for column in PRODUCT_INVESTIGATION_COLUMNS:
    count, percentage = missing_count_and_percentage(products, column)
    if column in PRODUCT_METADATA_COLUMNS:
        classification = "Needs Investigation"
        evidence = (
            "Descriptive fields show coordinated absence; compare the all-four-missing pattern."
        )
        action = "Retain; document missing indicator and consider later train-only treatment"
    else:
        classification = "Needs Investigation"
        evidence = "All four physical measurements are absent together for a very small product set."
        action = "Investigate source records; consider later train-only treatment"
    findings_rows.append(
        {
            "Dataset": DATASET_FILES["Products"],
            "Column": column,
            "Missing Count": count,
            "Missing Percentage": percentage,
            "Percentage Basis": "Rows",
            "Classification": classification,
            "Evidence": evidence,
            "ML Impact": "Potential predictive feature availability",
            "Proposed Action": action,
        }
    )

for column in REVIEW_TEXT_COLUMNS:
    count, percentage = missing_count_and_percentage(reviews, column)
    findings_rows.append(
        {
            "Dataset": DATASET_FILES["Reviews"],
            "Column": column,
            "Missing Count": count,
            "Missing Percentage": percentage,
            "Percentage Basis": "Rows",
            "Classification": "Expected",
            "Evidence": (
                f"Comment text is optional; valid review-score records exist without it. "
                f"A further {int(review_hidden_missing.set_index('Column').loc[column, ['Whitespace Only', 'Suspected Placeholder']].sum())} "
                "populated values are whitespace-only or placeholder-like."
            ),
            "ML Impact": "None as a feature; retrospective analysis only",
            "Proposed Action": "Retain missingness; exclude all review data from purchase-time features",
        }
    )

for dataset_name, dataframe in OTHER_DATASETS.items():
    profile = hidden_missing_details.loc[
        hidden_missing_details["Dataset"].eq(dataset_name)
    ]
    actual_missing = int(profile["Pandas NaN"].sum())
    # Empty strings are excluded here because they are the same cells Pandas
    # already counts as NaN; only additional hidden representations are added.
    hidden_missing = int(
        profile[["Whitespace Only", "Suspected Placeholder", "Converted To NaN"]]
        .sum()
        .sum()
    )
    findings_rows.append(
        {
            "Dataset": DATASET_FILES[dataset_name],
            "Column": "All columns / suitable text columns",
            "Missing Count": actual_missing,
            "Missing Percentage": (
                actual_missing / dataframe.size * 100 if dataframe.size else 0.0
            ),
            "Percentage Basis": "Cells",
            "Classification": "Expected" if actual_missing == 0 and hidden_missing == 0 else "Needs Investigation",
            "Evidence": (
                f"Column scan found {actual_missing} Pandas NaN cells and {hidden_missing} hidden/suspected placeholder matches."
            ),
            "ML Impact": "Potential feature availability if any missingness is found",
            "Proposed Action": "Retain source values; investigate tokens in column context",
        }
    )

findings = pd.DataFrame(findings_rows)
display(findings)

,Dataset,Column,Missing Count,Missing Percentage,Percentage Basis,Classification,Evidence,ML Impact,Proposed Action
0,olist_orders_dataset.csv,order_approved_at,160,0.1609,Rows,Expected by some statuses / Needs Investigation when delivered,Missingness varies by order_status; 14 delivered rows are missing this field.,None as a feature; lifecycle-quality context only,Retain for audit; investigate delivered-status contradictions
1,olist_orders_dataset.csv,order_delivered_carrier_date,1783,1.7930,Rows,Expected by some statuses / Needs Investigation when delivered,Missingness varies by order_status; 2 delivered rows are missing this field.,None as a feature; lifecycle-quality context only,Retain for audit; investigate delivered-status contradictions
2,olist_orders_dataset.csv,order_delivered_customer_date,2965,2.9817,Rows,Expected by some statuses / Needs Investigation when delivered,Missingness varies by order_status; 8 delivered rows are missing this field.,Classification and regression target eligibility,Assess label eligibility; investigate delivered-status and canceled-with-delivery contradictions
3,olist_orders_dataset.csv,order_status vs order_delivered_customer_date,6,0.0060,Rows,Needs Investigation,"6 canceled orders have an actual customer-delivery timestamp (count of contradictions, not missi...",Open target-eligibility decision: timestamp-only vs status-based rule,Decide and document the eligibility rule in target construction; report the count
4,olist_products_dataset.csv,product_category_name,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
5,olist_products_dataset.csv,product_name_lenght,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
6,olist_products_dataset.csv,product_description_lenght,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
7,olist_products_dataset.csv,product_photos_qty,610,1.8512,Rows,Needs Investigation,Descriptive fields show coordinated absence; compare the all-four-missing pattern.,Potential predictive feature availability,Retain; document missing indicator and consider later train-only treatment
8,olist_products_dataset.csv,product_weight_g,2,0.0061,Rows,Needs Investigation,All four physical measurements are absent together for a very small product set.,Potential predictive feature availability,Investigate source records; consider later train-only treatment
9,olist_products_dataset.csv,product_length_cm,2,0.0061,Rows,Needs Investigation,All four physical measurements are absent together for a very small product set.,Potential predictive feature availability,Investigate source records; consider later train-only treatment


### Recommendations

1. Keep lifecycle missingness conditional on `order_status`; do not label every undelivered order as defective.
2. Investigate delivered-status orders missing actual delivery before target construction, and report all target-eligibility exclusions.
3. Record an explicit eligibility rule for canceled orders that carry an actual customer-delivery timestamp (timestamp-only vs status-based) during target construction, with the affected count.
4. Preserve the coordinated product-metadata pattern as evidence for later feature-treatment design; do not fill it in this task.
5. Investigate the isolated products without physical measurements before deciding on training-only preprocessing.
6. Treat review text as optional retrospective data and exclude the entire review source from purchase-time features.
7. Preserve distinctions among `NaN`, blank strings, whitespace, and suspected placeholder tokens; interpret literal categories in business context. Check hidden missingness against raw file text, because the default CSV loader converts common NA tokens before any string check can see them.

## 11. Learning Questions

These questions are intentionally unanswered.

1. Why is a missing value not automatically a data-quality problem?
2. Why should missing delivery dates be investigated against order status?
3. What is the difference between expected and suspicious missingness?
4. Why should we not blindly call `dropna()`?
5. How can missing target values affect training eligibility?
6. How is a missing model feature different from a missing target?
7. Why should review information not be used for purchase-time prediction?
8. What can patterns of missing values reveal about data collection?